# Stronger Baselines for Time Series Classification

This notebook evaluates stronger deep learning baselines: InceptionTime, DisjointCNN, and LITEMVTime.

In [ ]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set random seed for reproducibility
np.random.seed(42)

## 1. Load and Prepare Data

In [ ]:
def load_npy_dataset(file_path):
    """
    Load and prepare numpy dataset for time series classification.
    """
    data = np.load(file_path, allow_pickle=True).item()

    X_train = data["train"]["X"]
    y_train = np.array([int(x) for x in data["train"]["y"]])
    X_test = data["test"]["X"]
    y_test = np.array([int(x) for x in data["test"]["y"]])

    print(f"Original shapes:")
    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"y_test shape: {y_test.shape}")

    # Convert from (B,C,T) to (B,T,C) for aeon
    if len(X_train.shape) == 3:
        X_train = np.transpose(X_train, (0, 2, 1))
        X_test = np.transpose(X_test, (0, 2, 1))
        print(f"\nConverted to aeon format (B,T,C):")
        print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

    print(f"Classes: {np.unique(y_train)}")
    return X_train, y_train, X_test, y_test

In [ ]:
from pathlib import Path

def find_project_root(target="Datasets"):
    current = Path.cwd()
    while not (current / target).exists():
        if current.parent == current:
            raise FileNotFoundError(f"'{target}' not found in any parent directory.")
        current = current.parent
    return current

project_root = find_project_root("Datasets")

dataset_name = "CMJ"
dataset_path = project_root / "Datasets" / f"{dataset_name}.npy"


In [ ]:


# Load the dataset
X_train, y_train, X_test, y_test = load_npy_dataset(dataset_path)

## 2. Import Deep Learning Models

In [ ]:
# Import deep learning classifiers from aeon v1.1.0
from aeon.classification.deep_learning import (
    InceptionTimeClassifier,
    DisjointCNNClassifier,
    LITETimeClassifier,
)

## 3. Define and Run Models

In [ ]:
def run_models(X_train, y_train, X_test, y_test, n_epochs=500):
    """
    Run deep learning models and return results
    """

    models = {
        "InceptionTime": InceptionTimeClassifier(
            n_epochs=n_epochs, batch_size=32, random_state=42, verbose=False
        ),
        "DisjointCNN": DisjointCNNClassifier(
            n_epochs=n_epochs, batch_size=32, random_state=42, verbose=False
        ),
        "LITEMVTime": LITETimeClassifier(
            use_litemv=True,
            n_epochs=n_epochs,
            batch_size=32,
            random_state=42,
            verbose=False,
        ),
    }

    results = {}

    for model_name, model in models.items():
        print(f"\n{'=' * 60}")
        print(f"Training {model_name}")
        print(f"{'=' * 60}")

        try:
            # Training
            print(f"Starting training...")
            start_time = time.time()
            model.fit(X_train, y_train)
            fit_time = time.time() - start_time

            # Get epochs trained
            try:
                epochs_trained = len(model.history.history["loss"])
                print(f"Trained for {epochs_trained} epochs (early stopping)")
            except:
                epochs_trained = n_epochs
                print(f"Trained for {epochs_trained} epochs (full training)")

            # Predictions
            print(f"Making predictions...")
            start_pred_time = time.time()
            y_pred_test = model.predict(X_test)
            pred_time = time.time() - start_pred_time

            # Accuracy
            test_accuracy = accuracy_score(y_test, y_pred_test)

            results[model_name] = {
                "test_accuracy": test_accuracy,
                "epochs_trained": epochs_trained,
                "fit_time": fit_time,
                "pred_time": pred_time,
                "y_pred": y_pred_test,
            }

            print(f"\n{model_name} Results:")
            print(f"  Test Accuracy: {test_accuracy:.4f}")
            print(f"  Epochs: {epochs_trained}")
            print(f"  Training Time: {fit_time:.2f}s")
            print(f"  Prediction Time: {pred_time:.4f}s")

        except Exception as e:
            print(f"\nERROR with {model_name}: {str(e)}")
            results[model_name] = {
                "error": str(e),
                "test_accuracy": 0.0,
                "fit_time": 0.0,
                "pred_time": 0.0,
                "epochs_trained": 0,
            }

    return results

## 4. Run Experiments

In [ ]:
# Run all models
print(f"Starting experiments on dataset: {os.path.basename(dataset_path)}")
print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")

results = run_models(X_train, y_train, X_test, y_test)

## 5. Results Analysis

In [ ]:
# Print summary table
print("\n" + "=" * 80)
print("SUMMARY TABLE")
print("=" * 80)
print(f"{'Model':<15} {'Accuracy':<10} {'Train(s)':<10} {'Pred(s)':<10}")
print("-" * 80)

for model_name, metrics in results.items():
    if "error" not in metrics:
        print(
            f"{model_name:<15} {metrics['test_accuracy']:<10.3f} "
            f"{metrics['fit_time']:<10.2f} "
            f"{metrics['pred_time']:<10.4f}"
        )
    else:
        print(f"{model_name:<15} ERROR: {metrics['error'][:50]}...")

In [ ]:
# Find and analyze best model
successful_models = {k: v for k, v in results.items() if "error" not in v}

if successful_models:
    best_model = max(
        successful_models.keys(), key=lambda x: results[x]["test_accuracy"]
    )

    print(f"\n{'=' * 50}")
    print(f"BEST MODEL: {best_model}")
    print(f"{'=' * 50}")
    print(f"Accuracy: {results[best_model]['test_accuracy']:.4f}")
    print(f"Training Time: {results[best_model]['fit_time']:.2f}s")
    print(f"Prediction Time: {results[best_model]['pred_time']:.4f}s")

    if "y_pred" in results[best_model]:
        print(f"\nClassification Report for {best_model}:")
        print(classification_report(y_test, results[best_model]["y_pred"]))
else:
    print("\nNo models completed successfully!")

## 6. Save Results

In [ ]:
import json
from datetime import datetime

# Create results directory
os.makedirs("results", exist_ok=True)

# Prepare results for JSON
json_results = {}
for model_name, metrics in results.items():
    json_results[model_name] = {k: v for k, v in metrics.items() if k != "y_pred"}

# Find best model for summary
best_model_name = None
best_accuracy = 0.0
if successful_models:
    best_model_name = max(
        successful_models.keys(), key=lambda x: results[x]["test_accuracy"]
    )
    best_accuracy = results[best_model_name]["test_accuracy"]

save_data = {
    "dataset": dataset_name,
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "data_shapes": {"train": list(X_train.shape), "test": list(X_test.shape)},
    "results": json_results,
    "best_model": best_model_name,
    "best_accuracy": best_accuracy,
    "total_models": len(results),
    "successful_models": len(successful_models),
}

# Save results
filename = f"results/stronger_baselines_{dataset_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(filename, "w") as f:
    json.dump(save_data, f, indent=4)

print(f"\nResults saved to: {filename}")
print(f"Summary: {len(successful_models)}/{len(results)} models completed successfully")